In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import optuna
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, accuracy_score

# ─── STEP 1: DOWNLOAD LATEST DATA ───────────────────────────────────────────

USEFUL_COLS = [
    'Date', 'HomeTeam', 'AwayTeam', 'Referee',
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG',
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC',
    'HF', 'AF', 'HY', 'AY', 'HR', 'AR',
]

SEASONS = {
    '2526': '2025/26', '2425': '2024/25', '2324': '2023/24',
    '2223': '2022/23', '2122': '2021/22', '2021': '2020/21',
    '1920': '2019/20', '1819': '2018/19', '1718': '2017/18',
    '1617': '2016/17', '1516': '2015/16', '1415': '2014/15',
}

dfs = []
for code, name in SEASONS.items():
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"
    try:
        temp = pd.read_csv(url, usecols=lambda c: c in USEFUL_COLS)
        temp['Season'] = name
        dfs.append(temp)
        print(f"✅ {name} → {len(temp)} matches")
    except Exception as e:
        print(f"❌ {name} → {e}")

df = pd.concat(dfs, ignore_index=True)
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
df = df.dropna(subset=['FTR']).sort_values('Date').reset_index(drop=True)

print(f"\nTotal matches: {len(df)}")
print(f"Latest match: {df['Date'].max().date()}")

In [ ]:
# ─── STEP 2: FEATURE ENGINEERING ────────────────────────────────────────────

# --- FORM LAST 5 ---
records = []
for _, row in df.iterrows():
    records.append({'Date': row['Date'], 'Team': row['HomeTeam'], 'Points': 3 if row['FTR'] == 'H' else (1 if row['FTR'] == 'D' else 0)})
    records.append({'Date': row['Date'], 'Team': row['AwayTeam'],  'Points': 3 if row['FTR'] == 'A' else (1 if row['FTR'] == 'D' else 0)})

team_df = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
team_df['form_last5'] = team_df.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

home_form = team_df[['Date', 'Team', 'form_last5']].rename(columns={'Team': 'HomeTeam', 'form_last5': 'home_form_last5'})
away_form = team_df[['Date', 'Team', 'form_last5']].rename(columns={'Team': 'AwayTeam', 'form_last5': 'away_form_last5'})
df = df.merge(home_form, on=['Date', 'HomeTeam'], how='left')
df = df.merge(away_form, on=['Date', 'AwayTeam'], how='left')

# --- GOALS AVG ---
goal_records = []
for _, row in df.iterrows():
    goal_records.append({'Date': row['Date'], 'Team': row['HomeTeam'], 'scored': row['FTHG'], 'conceded': row['FTAG']})
    goal_records.append({'Date': row['Date'], 'Team': row['AwayTeam'], 'scored': row['FTAG'], 'conceded': row['FTHG']})

goals_df = pd.DataFrame(goal_records).sort_values('Date').reset_index(drop=True)
goals_df['avg_scored_last5']   = goals_df.groupby('Team')['scored'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
goals_df['avg_conceded_last5'] = goals_df.groupby('Team')['conceded'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

home_goals = goals_df[['Date', 'Team', 'avg_scored_last5', 'avg_conceded_last5']].rename(columns={'Team': 'HomeTeam', 'avg_scored_last5': 'home_avg_scored', 'avg_conceded_last5': 'home_avg_conceded'})
away_goals = goals_df[['Date', 'Team', 'avg_scored_last5', 'avg_conceded_last5']].rename(columns={'Team': 'AwayTeam', 'avg_scored_last5': 'away_avg_scored', 'avg_conceded_last5': 'away_avg_conceded'})
df = df.merge(home_goals, on=['Date', 'HomeTeam'], how='left')
df = df.merge(away_goals, on=['Date', 'AwayTeam'], how='left')

# --- REFEREE ---
referee_stats = []
for idx, row in df.iterrows():
    ref  = row['Referee']
    date = row['Date']
    past = df[(df['Referee'] == ref) & (df['Date'] < date)]
    if len(past) == 0:
        referee_stats.append({'ref_matches': 0, 'ref_home_win_rate': None, 'ref_yellows_avg': None, 'ref_reds_avg': None, 'ref_goals_avg': None})
    else:
        referee_stats.append({
            'ref_matches':       len(past),
            'ref_home_win_rate': (past['FTR'] == 'H').sum() / len(past),
            'ref_yellows_avg':   (past['HY'] + past['AY']).mean(),
            'ref_reds_avg':      (past['HR'] + past['AR']).mean(),
            'ref_goals_avg':     (past['FTHG'] + past['FTAG']).mean(),
        })

ref_df = pd.DataFrame(referee_stats)
df = pd.concat([df, ref_df], axis=1)

# --- ELO ---
K = 32
BASE_ELO = 1500
elo_dict = {}
elo_records = []

for _, row in df.iterrows():
    home, away = row['HomeTeam'], row['AwayTeam']
    if home not in elo_dict: elo_dict[home] = BASE_ELO
    if away not in elo_dict: elo_dict[away] = BASE_ELO

    elo_home, elo_away = elo_dict[home], elo_dict[away]
    expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))

    if row['FTR'] == 'H':   actual_home, actual_away = 1, 0
    elif row['FTR'] == 'A': actual_home, actual_away = 0, 1
    else:                   actual_home, actual_away = 0.5, 0.5

    elo_records.append({'Date': row['Date'], 'HomeTeam': home, 'AwayTeam': away, 'elo_home': elo_home, 'elo_away': elo_away, 'elo_diff': elo_home - elo_away})
    elo_dict[home] = elo_home + K * (actual_home - expected_home)
    elo_dict[away] = elo_away + K * (actual_away - (1 - expected_home))

elo_df = pd.DataFrame(elo_records)
df = df.merge(elo_df, on=['Date', 'HomeTeam', 'AwayTeam'], how='left')

# --- H2H ---
h2h_records = []
for _, row in df.iterrows():
    home, away, date = row['HomeTeam'], row['AwayTeam'], row['Date']
    past = df[
        (df['Date'] < date) &
        (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
         ((df['HomeTeam'] == away) & (df['AwayTeam'] == home)))
    ].tail(5)

    if len(past) == 0:
        h2h_records.append({'h2h_matches': 0, 'h2h_home_wins': None, 'h2h_draws': None, 'h2h_goals_avg': None})
    else:
        home_wins = sum(1 for _, p in past.iterrows() if (p['HomeTeam'] == home and p['FTR'] == 'H') or (p['HomeTeam'] == away and p['FTR'] == 'A'))
        draws = (past['FTR'] == 'D').sum()
        h2h_records.append({
            'h2h_matches':   len(past),
            'h2h_home_wins': home_wins / len(past),
            'h2h_draws':     draws / len(past),
            'h2h_goals_avg': (past['FTHG'] + past['FTAG']).mean(),
        })

h2h_df = pd.DataFrame(h2h_records)
df = pd.concat([df, h2h_df], axis=1)

print("✅ Feature engineering done")
print(df.shape)

In [ ]:
# ─── STEP 3: RETROSPECTIVE EVALUATION (TEST SET NATURAL) ────────────────────
# Before retraining, evaluate the CURRENT model on NEW matches.
# These matches were never seen during training or hyperparameter tuning,
# so they act as an honest, ongoing test set.

FEATURE_COLS = [
    'home_form_last5', 'away_form_last5',
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'ref_matches', 'ref_home_win_rate',
    'ref_yellows_avg', 'ref_reds_avg', 'ref_goals_avg',
    'elo_home', 'elo_away', 'elo_diff',
    'h2h_matches', 'h2h_home_wins', 'h2h_draws', 'h2h_goals_avg',
]

EXTRA_COLS = ['Referee', 'FTHG', 'FTAG', 'HY', 'AY', 'HR', 'AR']

df['target'] = df['FTR'].map({'H': 0, 'D': 1, 'A': 2})
df_model = df[
    FEATURE_COLS + EXTRA_COLS + ['target', 'Date', 'HomeTeam', 'AwayTeam', 'Season']
].dropna(subset=FEATURE_COLS).reset_index(drop=True)

# --- Evaluate current model on matches it never saw ---
model_path = 'data/model.pkl'
features_path = 'data/features.csv'
tracking_path = 'data/tracking.csv'

if os.path.exists(model_path) and os.path.exists(features_path):
    # Build set of (date, home, away) keys from OLD data
    old_features = pd.read_csv(features_path, parse_dates=['Date'])
    old_keys = set(
        old_features.apply(
            lambda r: (str(r['Date'].date()), r['HomeTeam'], r['AwayTeam']), axis=1
        )
    )

    # Identify NEW matches not present in old features
    df_model['_key'] = df_model.apply(
        lambda r: (str(r['Date'].date()), r['HomeTeam'], r['AwayTeam']), axis=1
    )
    new_mask = ~df_model['_key'].isin(old_keys)
    new_matches = df_model[new_mask].copy()
    df_model = df_model.drop(columns=['_key'])

    if len(new_matches) > 0:
        with open(model_path, 'rb') as f:
            old_model = pickle.load(f)

        X_new = new_matches[FEATURE_COLS].values
        y_new = new_matches['target'].values

        probs = old_model.predict_proba(X_new)
        preds = old_model.predict(X_new)

        ll = log_loss(y_new, probs)
        acc = accuracy_score(y_new, preds)

        # Log to tracking file
        entry = pd.DataFrame([{
            'eval_date':    datetime.now().strftime('%Y-%m-%d'),
            'new_matches':  len(new_matches),
            'log_loss':     round(ll, 4),
            'accuracy':     round(acc, 4),
            'period_start': str(new_matches['Date'].min().date()),
            'period_end':   str(new_matches['Date'].max().date()),
        }])

        if os.path.exists(tracking_path):
            tracking = pd.read_csv(tracking_path)
            tracking = pd.concat([tracking, entry], ignore_index=True)
        else:
            tracking = entry

        tracking.to_csv(tracking_path, index=False)

        print(f"📊 Retrospective evaluation on {len(new_matches)} NEW matches:")
        print(f"   Log-loss:  {ll:.4f}")
        print(f"   Accuracy:  {acc:.4f} ({sum(preds == y_new)}/{len(y_new)})")
        print(f"   Period:    {new_matches['Date'].min().date()} → {new_matches['Date'].max().date()}")
        print(f"   ✅ Saved to {tracking_path}")

        # Show individual predictions
        label_map = {0: 'H', 1: 'D', 2: 'A'}
        display_df = new_matches[['Date', 'HomeTeam', 'AwayTeam']].copy()
        display_df['actual'] = [label_map[v] for v in y_new]
        display_df['pred']   = [label_map[v] for v in preds]
        display_df['hit']    = ['✅' if a == p else '❌' for a, p in zip(y_new, preds)]
        display_df['P(H)']   = probs[:, 0].round(3)
        display_df['P(D)']   = probs[:, 1].round(3)
        display_df['P(A)']   = probs[:, 2].round(3)
        print('\n' + display_df.to_string(index=False))
    else:
        print("ℹ️  No new matches found — nothing to evaluate")
else:
    print("ℹ️  First run — no previous model to evaluate against")

print(f"\n✅ Data prepared: {len(df_model)} rows")

In [ ]:
# ─── STEP 4: SAVE FEATURES & RETRAIN ────────────────────────────────────────

df_model.to_csv('data/features.csv', index=False)
print(f"✅ Features saved: {len(df_model)} rows")

X = df_model[FEATURE_COLS].values
y = df_model['target'].values

# Quick tuning (20 trials, faster than full 50)
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'eval_metric':      'mlogloss',
        'random_state':     42,
    }
    tscv = TimeSeriesSplit(n_splits=5)
    losses = []
    for train_idx, val_idx in tscv.split(X):
        m = XGBClassifier(**params)
        m.fit(X[train_idx], y[train_idx], verbose=False)
        losses.append(log_loss(y[val_idx], m.predict_proba(X[val_idx])))
    return np.mean(losses)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"✅ Best log_loss: {study.best_value:.4f}")

# Train final model
best_params = study.best_params
best_params['eval_metric'] = 'mlogloss'
best_params['random_state'] = 42

# FIX: Use TimeSeriesSplit for calibration instead of random KFold.
# Random KFold leaks future data into the calibration step.
calibrated_model = CalibratedClassifierCV(
    XGBClassifier(**best_params),
    method='isotonic',
    cv=TimeSeriesSplit(n_splits=3)
)
calibrated_model.fit(X, y)

with open('data/model.pkl', 'wb') as f:
    pickle.dump(calibrated_model, f)

print(f"✅ Model retrained and saved")
print(f"✅ Latest data: {df_model['Date'].max().date()}")

In [ ]:
# ─── STEP 5: SHOW TRACKING HISTORY ──────────────────────────────────────────
# View accumulated real-world performance over time.

if os.path.exists('data/tracking.csv'):
    tracking = pd.read_csv('data/tracking.csv')
    print("📈 Model performance tracking (real-world, unseen matches):")
    print(tracking.to_string(index=False))
    print(f"\n   Cumulative avg log_loss:  {tracking['log_loss'].mean():.4f}")
    print(f"   Cumulative avg accuracy:  {tracking['accuracy'].mean():.4f}")
    print(f"   Total evaluated matches:  {tracking['new_matches'].sum()}")
else:
    print("ℹ️  No tracking data yet — run this notebook again after new matches are played")